In [1]:
# Install the COCO annotation utilities required by Raster2Seq.
%pip install pycocotools

# Install dependencies used by Detectron2 within Raster2Seq.
%pip install fvcore omegaconf hydra-core cloudpickle

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# import geopandas as gpd
# ------------------------------------------------------------
# 1. Python standard libraries
# ------------------------------------------------------------
import sys         # Configure Python's module search paths.
import gzip, json  # Read compressed GeoJSON files and parse JSON.
import math        # Mathematical utilities; currently unused in this notebook.


# ------------------------------------------------------------
# 2. Numerical computing and deep learning libraries
# ------------------------------------------------------------
import numpy as np  # Store and manipulate polygon coordinates as arrays.
import torch        # PyTorch tensors used by Raster2Seq's preprocessing code.


# ------------------------------------------------------------
# 3. Make local source code available for imports
# Paths are relative to the notebook's current working directory.
# ------------------------------------------------------------

# Assumes Raster2Seq and this project are sibling folders.
sys.path.insert(0, "../Raster2Seq")

# Folder containing the map rendering utilities.
sys.path.insert(0, "data/osm_carto_compare")


# ------------------------------------------------------------
# 4. Map rendering utilities
# Source: data/osm_carto_compare/render_carto.py
# ------------------------------------------------------------

# lonlat_to_px: convert longitude/latitude to pixel coordinates at a given zoom.
# pixel_window: calculate pixel bounds for a geographic bounding box.
from render_carto import lonlat_to_px, pixel_window


# ------------------------------------------------------------
# 5. Preprocessing and visualization utilities from Raster2Seq
# ------------------------------------------------------------

# Source: Raster2Seq/datasets/data_utils.py
# sort_polygons: arrange polygons in a consistent spatial order.
# plot_polygons: visualize a list of polygons.
from datasets.data_utils import sort_polygons, plot_polygons

# Source: Raster2Seq/datasets/discrete_tokenizer.py
# Encode quantized coordinates and special markers as token IDs.
# Not called directly in this notebook; used internally by ConvertToCocoDict.
from datasets.discrete_tokenizer import DiscreteTokenizer

# Source: Raster2Seq/datasets/poly_data.py
# Prepare annotations for Raster2Seq.
# Here, we use its sequence conversion method to produce coordinate targets
# and labels identifying coordinates, polygon separators, and sequence endings.
from datasets.poly_data import ConvertToCocoDict

# Source: Raster2Seq/util/plot_utils.py
# Drawing utilities for annotations and floorplans with semantic labels.
# Currently unused in this notebook.
# These visualization functions are not used in this notebook.
# from util.plot_utils import plot_anno, plot_semantic_rich_floorplan_opencv


# ------------------------------------------------------------
# 6. Image loading and plotting
# ------------------------------------------------------------
from matplotlib import pyplot as plt

# Matplotlib's polygon drawing class.
# The alias distinguishes it from other libraries' Polygon classes.
from matplotlib.patches import Polygon as MplPolygon

from PIL import Image  # Load raster images, such as PNG map crops.

# Jupyter magic: display Matplotlib figures inside the notebook.
%matplotlib inline

In [3]:
# Load one map crop from a compressed GeoJSON file.
with gzip.open(
    "crops/staten_island_14645.geojson.gz",
    "rt",
    encoding="utf-8"
) as fh:
    data = json.load(fh)

# Confirm that the file loaded successfully.
print("Loaded features:", len(data["features"]))

Loaded features: 65


In [4]:
# Inspect the geographic bounding box and map metadata.
data["bbox"], data["metadata"];

In [5]:
# Use the zoom level used to render the PNG map.
zoom = data["metadata"]["camera_zoom"]

# Convert geographic bounds to global map pixel coordinates.
# Helper source: data/osm_carto_compare/render_carto.py
# (x0, y0): top-left corner of the crop.
# (x1, y1): bottom-right corner of the crop.
x0, y0, x1, y1 = pixel_window(data["bbox"], zoom)

# Calculate the crop's width and height in map pixel coordinates.
W, H = x1 - x0, y1 - y0


def to_px(lon, lat):
    """Convert longitude/latitude to pixel coordinates within this crop."""

    # Convert geographic coordinates to global map pixel coordinates.
    px, py = lonlat_to_px(lon, lat, zoom)

    # Shift the origin to the crop's top-left corner.
    # Local x increases to the right; local y increases downward.
    return (px - x0, py - y0)


# Display the coordinate window size for inspection.
print(f"Map pixel window: {W} × {H}, zoom: {zoom}")

Map pixel window: 553 × 594, zoom: 18


In [6]:
# Extract polygon exterior rings from GeoJSON.
# Each output item contains (pixel_coordinates, layer_name).
polygons = []

for feature in data["features"]:
    geometry = feature["geometry"]

    # Skip features without geometry.
    if geometry is None:
        continue

    if geometry["type"] == "Polygon":
        # The first ring is the exterior boundary.
        # Any additional rings represent holes and are ignored here.
        outer_rings = [geometry["coordinates"][0]]

    elif geometry["type"] == "MultiPolygon":
        # Extract one exterior ring from each polygon component.
        # Store each component as a separate polygon.
        outer_rings = [
            part[0] for part in geometry["coordinates"]
        ]

    else:
        # Skip points, lines, and other geometry types for this experiment.
        continue

    for outer_ring in outer_rings:
        poly_px = []

        # GeoJSON rings repeat the first vertex at the end.
        # Remove that duplicate closing vertex.
        for lon, lat in outer_ring[:-1]:

            # Convert to crop-local pixels using the function defined above.
            poly_px.append(to_px(lon, lat))

        # Preserve the source layer, such as "building" or "landuse".
        layer = feature["properties"]["omt_layer"]

        # Keep this polygon's coordinates together with its layer name.
        polygons.append((poly_px, layer))

print("Extracted polygons:", len(polygons))

Extracted polygons: 178


In [7]:
# Install Shapely for geometric operations.
%pip install shapely

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Geometry utilities from the external Shapely library.
from shapely.geometry import LineString, box
from shapely.ops import polygonize, unary_union

# Optional experiment: create enclosed regions from transportation
# and waterway lines, then store their exterior coordinates.
lines = []

for feature in data["features"]:
    geometry = feature["geometry"]

    if geometry is None:
        continue

    # Select transportation and waterway features.
    layer = feature["properties"]["omt_layer"]
    if layer not in ("transportation", "waterway"):
        continue

    # Treat both geometry types as a list of individual lines.
    if geometry["type"] == "LineString":
        parts = [geometry["coordinates"]]
    elif geometry["type"] == "MultiLineString":
        parts = geometry["coordinates"]
    else:
        continue

    for part in parts:
        # Convert geographic coordinates to crop-local pixel coordinates.
        points = [to_px(lon, lat) for lon, lat in part]

        # A LineString requires at least two coordinate positions.
        if len(points) >= 2:
            lines.append(LineString(points))

# Create a rectangle representing the map crop.
crop_bounds = box(0, 0, W, H)

# Merge the linework and remove portions outside the crop.
clipped_lines = unary_union(lines).intersection(crop_bounds)

# Experimental minimum area, measured in square map-pixel units.
min_block_area = 20.0

if clipped_lines.is_empty or clipped_lines.length == 0:
    # Avoid generating a whole-image polygon when no usable lines exist.
    blocks = []
else:
    # Include the image border so regions can close along the crop edge.
    # Union also splits crossing lines at their intersection points.
    connected_lines = unary_union([
        clipped_lines,
        crop_bounds.exterior
    ])

    # Create polygons from enclosed regions and filter out tiny regions.
    blocks = [
        block
        for block in polygonize(connected_lines)
        if block.area > min_block_area
    ]

# Keep exterior vertices only and remove the repeated closing vertex.
# "block" is an experimental label assigned here.
block_polygons = [
    (list(block.exterior.coords)[:-1], "block")
    for block in blocks
]

print("Generated block polygons:", len(block_polygons))

Generated block polygons: 64


In [9]:
# Extract each polygon's vertex list.
# The underscore means the layer label is not used in this expression.
points_only = [coords for coords, _ in polygons]

# Source: Raster2Seq/datasets/data_utils.py
# Use each polygon's topmost vertex as its sorting reference.
# If multiple vertices share the minimum y, choose the leftmost one.
#
# Group polygons into rows using a y-coordinate tolerance of 20 pixels.
# Process rows from top to bottom, and polygons within each row
# from left to right.
sorted_polys, order = sort_polygons(
    points_only,
    tolerance=20
)

# "order" contains the polygons' original indices after sorting.
# Reorder the layer labels to keep them matched with their polygons.
sorted_layers = [polygons[i][1] for i in order]

# Preview the result without printing every coordinate.
print("Sorted polygons:", len(sorted_polys))
print("First 5 original indices:", order[:5])
print("First 5 layer labels:", sorted_layers[:5])

# Count target positions for the current converter settings:
# one position per vertex, plus one separator/end position per polygon.
# Assumes no additional class tokens.
num_vertices = sum(len(poly) for poly in sorted_polys)
required_seq_len = num_vertices + len(sorted_polys)

print("Total vertices:", num_vertices)
print("Required target sequence length:", required_seq_len)

Sorted polygons: 178
First 5 original indices: [124, 123, 35, 100, 50]
First 5 layer labels: ['building', 'building', 'building', 'building', 'building']
Total vertices: 1036
Required target sequence length: 1214


In [10]:
# Geometry tools from Shapely.
from shapely.geometry import Polygon, box

# Source: Raster2Seq/util/poly_ops.py
# Standardize each polygon's starting vertex and traversal direction.
from util.poly_ops import resort_corners

# Pixel-center bounds, matching normalization by (longer_side - 1).
crop_bounds = box(0, 0, W - 1, H - 1)
visible = []
skipped_invalid = 0

# Process objects separately so overlapping polygons remain separate.
for coords, layer in zip(sorted_polys, sorted_layers):
    if len(coords) < 3:
        skipped_invalid += 1
        continue

    polygon = Polygon(coords)

    # Exclude invalid outlines for this initial experiment.
    if not polygon.is_valid:
        skipped_invalid += 1
        continue

    # Clipping may split one polygon into multiple visible components.
    pending = [polygon.intersection(crop_bounds)]

    while pending:
        part = pending.pop()

        if part.is_empty:
            continue

        if part.geom_type in ("MultiPolygon", "GeometryCollection"):
            pending.extend(part.geoms)
            continue

        # Ignore results that contain no polygon area.
        if part.geom_type != "Polygon" or part.area <= 0:
            continue

        # Keep the exterior only; remove the repeated closing vertex.
        vertices = np.asarray(
            part.exterior.coords[:-1], dtype=np.float64
        )

        vertices = resort_corners(vertices).reshape(-1, 2)
        visible.append((vertices, layer))

if not visible:
    raise ValueError("No usable polygons remain inside the image.")

# Re-sort because clipping can change positions or split objects.
visible_polys, visible_order = sort_polygons(
    [coords for coords, _ in visible],
    tolerance=20
)
visible_layers = [visible[i][1] for i in visible_order]

print("Polygons before clipping:", len(sorted_polys))
print("Visible polygon components:", len(visible_polys))
print("Skipped invalid outlines:", skipped_invalid)

Polygons before clipping: 178
Visible polygon components: 146
Skipped invalid outlines: 0


In [11]:
# Build class IDs for this single-sample experiment.
# Before multi-image training, use one fixed mapping for the whole dataset.
layer_ids = {
    name: i for i, name in enumerate(sorted(set(sorted_layers)))
}
labels = [layer_ids[name] for name in visible_layers]

# Input encoding grid; coordinate targets remain continuous.
num_bins = 64

# Sequence capacity for this sample.
seq_len = 1536

# Use one scale to preserve the polygons' aspect ratio.
# This assumes a square canvas padded on the right and bottom.
# The PNG preprocessing must use the same coordinate convention.
normalization_scale = max(W, H) - 1
norm_polys = [
    coords / normalization_scale
    for coords in visible_polys
]

# One target position per vertex, plus one SEP/EOS per polygon.
num_vertices = sum(len(coords) for coords in norm_polys)
required_seq_len = num_vertices + len(norm_polys)

# Allow one spare slot for the tokenizer's temporary start marker.
if required_seq_len + 1 > seq_len:
    raise ValueError(
        f"Sequence capacity {seq_len} is too small: "
        f"{required_seq_len} target positions plus construction overhead."
    )

# Source: Raster2Seq/datasets/poly_data.py
# Call the sequence method directly; no COCO annotation file is loaded.
conv = ConvertToCocoDict(
    root="",
    augmentations=None,
    image_norm=False,
    poly2seq=True,
    semantic_classes=len(layer_ids) + 1,  # Reserve an undefined class ID.
    add_cls_token=False,
    per_token_class=False,              # One semantic label per polygon.
    num_bins=num_bins,
    seq_len=seq_len,
)

out = conv._get_bilinear_interpolation_coeffs(norm_polys, labels)

# Check that targets and encoded inputs retain the full prepared sample.
assert int((out["token_labels"] == 0).sum().item()) == num_vertices
assert int(
    (out["seq11"] != conv.tokenizer.pad).sum().item()
) == required_seq_len

print("Layer IDs:", layer_ids)
print("Coordinate targets:", num_vertices)
print("Used / allocated positions:", required_seq_len, "/", seq_len)
print("Target tensor shape:", tuple(out["target_seq"].shape))

Layer IDs: {'building': 0, 'landcover': 1, 'landuse': 2, 'water': 3}
Coordinate targets: 856
Used / allocated positions: 1002 / 1536
Target tensor shape: (1536, 2)
